# 00 — Preprocessing pipeline (CASA0006 VRU project)

This notebook produces `data/clean/vru_clean.csv` from raw STATS19 + boundary +
IMD 2019 sources. **It is not part of the submission.** The submitted
`analysis.ipynb` reads `vru_clean.csv` from a public GitHub raw URL, satisfying
the CASA0006 1-hour Restart-and-Run-All constraint.

**Inputs**
- `data/raw/dft-road-casualty-statistics-{collision,casualty,vehicle}-{2022,2023,2024}.csv`
- `data/boundaries/statistical-gis-boundaries-london/ESRI/LSOA_2011_London_gen_MHW.shp`
- `data/boundaries/statistical-gis-boundaries-london/ESRI/London_Borough_Excluding_MHW.shp`
- `data/external/File_7_-_All_IoD2019_Scores__Ranks__Deciles_and_Population_Denominators_3.csv`

**Output**
- `data/clean/vru_clean.csv` — one row per VRU casualty, 2022–2024, Greater London only.

In [1]:
"""Preprocessing pipeline — see notebook header."""

from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd

# ---------------------------------------------------------------------------
# Paths (resolved relative to this notebook: notebooks/00_preprocessing.ipynb)
# ---------------------------------------------------------------------------
PROJECT_ROOT = Path.cwd().parent          # casa0006-vru-london/
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
BOUNDARY_DIR = DATA_DIR / "boundaries" / "statistical-gis-boundaries-london" / "ESRI"
EXTERNAL_DIR = DATA_DIR / "external"
CLEAN_DIR = DATA_DIR / "clean"
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------------
# Source files
# ---------------------------------------------------------------------------
YEARS = (2022, 2023, 2024)

COLLISION_FILES = [RAW_DIR / f"dft-road-casualty-statistics-collision-{y}.csv" for y in YEARS]
CASUALTY_FILES  = [RAW_DIR / f"dft-road-casualty-statistics-casualty-{y}.csv"  for y in YEARS]
VEHICLE_FILES   = [RAW_DIR / f"dft-road-casualty-statistics-vehicle-{y}.csv"   for y in YEARS]

LSOA_SHP    = BOUNDARY_DIR / "LSOA_2011_London_gen_MHW.shp"
BOROUGH_SHP = BOUNDARY_DIR / "London_Borough_Excluding_MHW.shp"
IMD_CSV     = EXTERNAL_DIR / "File_7_-_All_IoD2019_Scores__Ranks__Deciles_and_Population_Denominators_3.csv"

CLEAN_CSV   = CLEAN_DIR / "vru_clean.csv"

# ---------------------------------------------------------------------------
# Sanity check — fail fast if any source is missing
# ---------------------------------------------------------------------------
required = COLLISION_FILES + CASUALTY_FILES + VEHICLE_FILES + [LSOA_SHP, BOROUGH_SHP, IMD_CSV]
missing = [str(p) for p in required if not p.exists()]
assert not missing, f"Missing source files:\n" + "\n".join(missing)

print(f"pandas    {pd.__version__}")
print(f"numpy     {np.__version__}")
print(f"geopandas {gpd.__version__}")
print(f"\nProject root: {PROJECT_ROOT}")
print(f"All {len(required)} source files present.")

pandas    2.3.3
numpy     1.26.4
geopandas 1.1.3

Project root: c:\Users\FanYuxiang\Desktop\casa0006-vru-london
All 12 source files present.


## Step 1 — Load 3-year collision data

Read 2022–2024 STATS19 collision CSVs, concat, and verify that the columns we will use downstream actually exist (fail fast if STATS19 silently renamed anything).

In [2]:
# Load 3-year collision data
collision_dfs = [pd.read_csv(p, low_memory=False) for p in COLLISION_FILES]
collisions = pd.concat(collision_dfs, ignore_index=True)
del collision_dfs  # free memory

print(f"Collisions loaded: {len(collisions):,} rows × {len(collisions.columns)} cols")
print(f"Memory: {collisions.memory_usage(deep=True).sum()/1e6:.1f} MB\n")

# DfT renamed `accident_*` → `collision_*` in the post-2022 STATS19 schema.
# The LSOA column kept the legacy "accident" prefix (DfT inconsistency).
expected_cols = ["collision_index", "collision_year", "collision_severity",
                 "latitude", "longitude", "lsoa_of_accident_location",
                 "date", "time"]
missing_cols = [c for c in expected_cols if c not in collisions.columns]
assert not missing_cols, (
    f"Missing expected columns: {missing_cols}\n"
    f"Actual columns: {list(collisions.columns)}"
)
print("Key columns present:", expected_cols)

print("\nYear distribution:")
print(collisions["collision_year"].value_counts().sort_index().to_string())

print("\nSeverity distribution (1=Fatal, 2=Serious, 3=Slight):")
print(collisions["collision_severity"].value_counts().sort_index().to_string())

Collisions loaded: 311,189 rows × 44 cols
Memory: 254.2 MB

Key columns present: ['collision_index', 'collision_year', 'collision_severity', 'latitude', 'longitude', 'lsoa_of_accident_location', 'date', 'time']

Year distribution:
collision_year
2022    106004
2023    104258
2024    100927

Severity distribution (1=Fatal, 2=Serious, 3=Slight):
collision_severity
1      4626
2     70338
3    236225


## Step 2 — Load 3-year casualty data

Casualty file is one row per casualty (one collision can have multiple casualties). Verify the merge key `collision_index` and the columns needed for VRU filtering (`casualty_class`, `casualty_type`) and downstream Logistic regression (`age_of_casualty`, `sex_of_casualty`).

In [3]:
# Load 3-year casualty data
casualty_dfs = [pd.read_csv(p, low_memory=False) for p in CASUALTY_FILES]
casualties = pd.concat(casualty_dfs, ignore_index=True)
del casualty_dfs

print(f"Casualties loaded: {len(casualties):,} rows × {len(casualties.columns)} cols")
print(f"Memory: {casualties.memory_usage(deep=True).sum()/1e6:.1f} MB\n")

# Verify columns we will use downstream.
# Merge key with collision = `collision_index` (DfT post-2022 rename also applies here).
expected_cols = ["collision_index", "casualty_severity",
                 "casualty_class", "casualty_type",
                 "age_of_casualty", "sex_of_casualty"]
missing_cols = [c for c in expected_cols if c not in casualties.columns]
assert not missing_cols, (
    f"Missing expected columns: {missing_cols}\n"
    f"Actual columns: {list(casualties.columns)}"
)
print("Key columns present:", expected_cols)

print("\nCasualty severity distribution (1=Fatal, 2=Serious, 3=Slight):")
print(casualties["casualty_severity"].value_counts().sort_index().to_string())

print("\nCasualty class (1=Driver/rider, 2=Passenger, 3=Pedestrian):")
print(casualties["casualty_class"].value_counts().sort_index().to_string())

print("\nCasualty type — top 10 (code 1 = Pedal cycle):")
print(casualties["casualty_type"].value_counts().head(10).to_string())

Casualties loaded: 396,729 rows × 23 cols
Memory: 143.2 MB

Key columns present: ['collision_index', 'casualty_severity', 'casualty_class', 'casualty_type', 'age_of_casualty', 'sex_of_casualty']

Casualty severity distribution (1=Fatal, 2=Serious, 3=Slight):
casualty_severity
1      4937
2     77890
3    313902

Casualty class (1=Driver/rider, 2=Passenger, 3=Pedestrian):
casualty_class
1    261434
2     77529
3     57766

Casualty type — top 10 (code 1 = Pedal cycle):
casualty_type
 9     212282
 0      57766
 1      45241
 3      26167
 5      13441
 19     11291
 11      6767
 4       5699
 8       4447
-1       3449


## Step 3 — Filter collisions to Greater London

Use the LSOA 2011 shapefile (data.london.gov.uk) as a London-membership whitelist. Any collision whose `lsoa_of_accident_location` matches a London LSOA code is kept. This is the most accurate London-boundary method available — points outside Greater London (and NA-LSOA collisions) are dropped.

In [4]:
# Load London LSOA boundary (2011 version, matches IMD 2019)
lsoa = gpd.read_file(LSOA_SHP)

print(f"LSOA shapefile: {len(lsoa):,} polygons")
print(f"Columns: {list(lsoa.columns)}")
print(f"CRS: {lsoa.crs}\n")

# The LSOA code column is conventionally LSOA11CD in 2011 boundaries.
# Pin it explicitly so a silent rename would fail loud.
assert "LSOA11CD" in lsoa.columns, f"Expected LSOA11CD column, got: {list(lsoa.columns)}"

london_lsoa_codes = set(lsoa["LSOA11CD"].unique())
print(f"London LSOA codes: {len(london_lsoa_codes):,}")  # Expect ~4,835
print(f"Sample codes: {sorted(london_lsoa_codes)[:3]}")  # All start with E01

# Filter collisions to London by LSOA whitelist
n_before = len(collisions)
n_na = collisions["lsoa_of_accident_location"].isna().sum()
collisions_lon = collisions[collisions["lsoa_of_accident_location"].isin(london_lsoa_codes)].copy()
n_after = len(collisions_lon)

print(f"\nCollisions: {n_before:,} → {n_after:,} (kept {n_after/n_before:.1%})")
print(f"Dropped: {n_before - n_after:,} (of which {n_na:,} were NA-LSOA)")

LSOA shapefile: 4,835 polygons
Columns: ['LSOA11CD', 'LSOA11NM', 'MSOA11CD', 'MSOA11NM', 'LAD11CD', 'LAD11NM', 'RGN11CD', 'RGN11NM', 'USUALRES', 'HHOLDRES', 'COMESTRES', 'POPDEN', 'HHOLDS', 'AVHHOLDSZ', 'geometry']
CRS: PROJCS["OSGB36 / British National Grid",GEOGCS["OSGB36",DATUM["Ordnance_Survey_of_Great_Britain_1936",SPHEROID["Airy 1830",6377563.396,299.3249646,AUTHORITY["EPSG","7001"]],AUTHORITY["EPSG","6277"]],PRIMEM["Greenwich",0],UNIT["Degree",0.0174532925199433]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",49],PARAMETER["central_meridian",-2],PARAMETER["scale_factor",0.999601272],PARAMETER["false_easting",400000],PARAMETER["false_northing",-100000],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]

London LSOA codes: 4,835
Sample codes: ['E01000001', 'E01000002', 'E01000003']

Collisions: 311,189 → 65,731 (kept 21.1%)
Dropped: 245,458 (of which 0 were NA-LSOA)


## Step 4 — Filter casualties to VRUs

Pedestrians (`casualty_class == 3`) and cyclists (`casualty_type == 1`) are the two VRU groups our research question targets. We tag each VRU casualty with a `vru_type` label so downstream EDA, choropleth maps, and SHAP plots can stratify by mode.

In [5]:
# Filter casualties to VRUs (pedestrians + cyclists)
# VRU definition (handoff §4): casualty_class == 3 (pedestrian) OR casualty_type == 1 (pedal cycle)
n_before = len(casualties)
vru_mask = (casualties["casualty_class"] == 3) | (casualties["casualty_type"] == 1)
vru_casualties = casualties[vru_mask].copy()

# Tag VRU subtype for downstream stratification / EDA
# (class==3 implies pedestrian; type==1 implies cyclist; mutually exclusive by STATS19 design)
vru_casualties["vru_type"] = np.where(
    vru_casualties["casualty_type"] == 1, "cyclist", "pedestrian"
)

n_after = len(vru_casualties)
print(f"Casualties: {n_before:,} → {n_after:,} VRUs (kept {n_after/n_before:.1%})\n")

print("VRU breakdown:")
print(vru_casualties["vru_type"].value_counts().to_string())

print("\nVRU severity (UK-wide; will further restrict to London after merge):")
print(vru_casualties["casualty_severity"].value_counts().sort_index().to_string())

Casualties: 396,729 → 103,007 VRUs (kept 26.0%)

VRU breakdown:
vru_type
pedestrian    57766
cyclist       45241

VRU severity (UK-wide; will further restrict to London after merge):
casualty_severity
1     1459
2    27665
3    73883


## Step 5 — Merge to build the analysis table `vru`

Inner-merge VRU casualties (UK-wide) with London collisions on `collision_index`. The inner join automatically restricts to London VRUs (any VRU casualty whose collision is outside London is dropped). The result is the master analysis table: **one row per London VRU casualty, with collision-level features attached**.

We explicitly select 7 casualty columns + 20 collision columns. Categorical encoding is **deferred to `analysis.ipynb`** — `vru_clean.csv` stays as raw integer codes so downstream feature engineering can choose its own encoder (OneHotEncoder, target encoding, etc.).

In [6]:
# Inner merge VRU casualties × London collisions on collision_index.
# This automatically restricts to London (only VRUs whose collision is in London),
# and attaches collision-level features (date, time, environmental, road geometry).

casualty_cols = ["collision_index", "casualty_severity", "casualty_class",
                 "casualty_type", "vru_type",
                 "age_of_casualty", "sex_of_casualty"]

collision_cols = [
    "collision_index",
    # Spatial / temporal
    "latitude", "longitude", "lsoa_of_accident_location",
    "date", "time", "day_of_week",
    # Borough (for spatial CV groups)
    "local_authority_ons_district",
    # Environmental
    "light_conditions", "weather_conditions", "road_surface_conditions",
    # Road geometry / context
    "speed_limit", "road_type", "junction_detail", "junction_control",
    "urban_or_rural_area", "pedestrian_crossing",
    # Collision context
    "number_of_vehicles", "number_of_casualties",
]

# Verify columns are present (fail fast on schema drift)
for src_name, df, cols in [("casualty", vru_casualties, casualty_cols),
                            ("collision", collisions_lon, collision_cols)]:
    missing = [c for c in cols if c not in df.columns]
    assert not missing, f"{src_name} table missing: {missing}"

# Inner merge — many casualties (left) per collision (right)
vru = vru_casualties[casualty_cols].merge(
    collisions_lon[collision_cols],
    on="collision_index",
    how="inner",
    validate="many_to_one",
)

print(f"London VRU casualties: {len(vru):,} rows × {len(vru.columns)} cols")
print(f"Memory: {vru.memory_usage(deep=True).sum()/1e6:.1f} MB\n")

print("VRU subtype × severity (counts):")
print(pd.crosstab(vru["vru_type"], vru["casualty_severity"], margins=True).to_string())

print("\nYearly distribution (parsed from collision date):")
print(pd.to_datetime(vru["date"], dayfirst=True).dt.year.value_counts().sort_index().to_string())

London VRU casualties: 27,478 rows × 25 cols
Memory: 15.1 MB

VRU subtype × severity (counts):
casualty_severity    1     2      3    All
vru_type                                  
cyclist             24  2838  11438  14300
pedestrian         154  3462   9562  13178
All                178  6300  21000  27478

Yearly distribution (parsed from collision date):
date
2022    9651
2023    9352
2024    8475


## Step 6 — Construct binary target + data quality checks

Build the binary outcome `severe = 1{casualty_severity ∈ {Fatal, Serious}}` and run sanity checks: NA rates, lat/lon bounding box (must lie inside Greater London), and STATS19 `-1` "missing/unknown" counts on the categorical features. Missing codes are **kept as-is** in `vru_clean.csv` so `analysis.ipynb` can decide column-by-column imputation later (median for numerics, dedicated "unknown" category for categoricals).

In [7]:
# Build the binary target: severe = (casualty_severity in {1, 2})
vru["severe"] = vru["casualty_severity"].isin([1, 2]).astype(int)

print(f"Severe rate (Fatal + Serious):")
print(f"  overall:    {vru['severe'].mean():.1%}  ({vru['severe'].sum():,} of {len(vru):,})")
print(f"  cyclist:    {vru[vru['vru_type']=='cyclist']['severe'].mean():.1%}")
print(f"  pedestrian: {vru[vru['vru_type']=='pedestrian']['severe'].mean():.1%}")

# NA rates per column
print(f"\nNA rates per column (only columns with NA shown):")
na_pct = (vru.isna().mean() * 100).round(2)
na_pct = na_pct[na_pct > 0]
print(na_pct.to_string() if len(na_pct) > 0 else "  (no NA in any column)")

# Coordinate sanity: London bbox ≈ lon ∈ [-0.51, 0.34], lat ∈ [51.28, 51.69]
print(f"\nLat/lon bbox:")
print(f"  longitude: [{vru['longitude'].min():.4f}, {vru['longitude'].max():.4f}]")
print(f"  latitude : [{vru['latitude'].min():.4f}, {vru['latitude'].max():.4f}]")

# STATS19 uses -1 for "data missing or out of range".
# Count -1 in our key categorical / numeric features (kept as-is; analysis.ipynb handles imputation).
neg_one_cols = ["light_conditions", "weather_conditions", "road_surface_conditions",
                "speed_limit", "road_type", "junction_detail", "junction_control",
                "urban_or_rural_area", "pedestrian_crossing",
                "age_of_casualty", "sex_of_casualty"]
print(f"\n-1 (missing/unknown) counts in features:")
for c in neg_one_cols:
    n = (vru[c] == -1).sum()
    if n > 0:
        print(f"  {c}: {n:,} ({n/len(vru):.2%})")

print(f"\nFinal shape: {vru.shape}")

Severe rate (Fatal + Serious):
  overall:    23.6%  (6,478 of 27,478)
  cyclist:    20.0%
  pedestrian: 27.4%

NA rates per column (only columns with NA shown):
  (no NA in any column)

Lat/lon bbox:
  longitude: [-0.4959, 0.3017]
  latitude : [51.2928, 51.6856]

-1 (missing/unknown) counts in features:
  junction_detail: 1,874 (6.82%)
  junction_control: 5,369 (19.54%)
  pedestrian_crossing: 1 (0.00%)
  age_of_casualty: 937 (3.41%)
  sex_of_casualty: 497 (1.81%)

Final shape: (27478, 26)


## Step 7 — Export `vru_clean.csv`

Final step: serialise the master VRU table to `data/clean/vru_clean.csv`. The submitted `analysis.ipynb` reads this file via a **GitHub raw URL** (per CASA0006 dataset-sharing rules), so the user must `git push` `data/clean/vru_clean.csv` to a public repo after this notebook runs successfully.

The read-back assertion verifies that what `analysis.ipynb` will load is byte-identical (in shape and columns) to the `vru` DataFrame built above — no schema drift between preprocessing and analysis.

In [8]:
# Export clean CSV — single source of truth for analysis.ipynb
vru.to_csv(CLEAN_CSV, index=False)

# Verify file written + report sizing
file_size_mb = CLEAN_CSV.stat().st_size / 1e6
print(f"Wrote: {CLEAN_CSV}")
print(f"Rows × cols: {vru.shape}")
print(f"File size: {file_size_mb:.2f} MB (under GitHub 100 MB single-file limit)\n")

# Show final dtypes (analysis.ipynb will use these for feature engineering)
print("Column dtypes:")
print(vru.dtypes.to_string())

# Quick read-back sanity check — exact same path analysis.ipynb will follow
echo = pd.read_csv(CLEAN_CSV)
assert echo.shape == vru.shape, "Read-back shape mismatch!"
assert (echo.columns == vru.columns).all(), "Read-back columns mismatch!"
print(f"\nRead-back OK: {echo.shape}")

Wrote: c:\Users\FanYuxiang\Desktop\casa0006-vru-london\data\clean\vru_clean.csv
Rows × cols: (27478, 26)
File size: 3.30 MB (under GitHub 100 MB single-file limit)

Column dtypes:
collision_index                  object
casualty_severity                 int64
casualty_class                    int64
casualty_type                     int64
vru_type                         object
age_of_casualty                   int64
sex_of_casualty                   int64
latitude                        float64
longitude                       float64
lsoa_of_accident_location        object
date                             object
time                             object
day_of_week                       int64
local_authority_ons_district     object
light_conditions                  int64
weather_conditions                int64
road_surface_conditions           int64
speed_limit                       int64
road_type                         int64
junction_detail                   int64
junction_control    

## Step 8 — Export London LSOA GeoJSON

The submitted `analysis.ipynb` needs LSOA polygons for choropleth + LISA cluster maps **and** for Queen contiguity in the spatial weights matrix. The original ESRI shapefile (~17 MB) lives in `data/boundaries/` (gitignored). Here we slim it to identifier columns and reproject to WGS84, but **keep the full geometry** — independent per-polygon simplification breaks shared boundaries between adjacent LSOAs and renders Queen contiguity inoperative. The resulting ~17 MB GeoJSON is well under GitHub's 100 MB single-file limit.

In [9]:
# Export London LSOA boundary GeoJSON
# Used by analysis.ipynb for choropleth + LISA maps + Queen contiguity weights
# AND for aggregating to MSOA / Borough scales (Section 1.7 MAUP analysis).
# Slim = LSOA + MSOA + LAD identifiers + ORIGINAL geometry, reprojected to WGS84.
#
# NOTE: We deliberately do NOT pre-simplify geometry. Per-polygon simplify()
# breaks shared boundaries between adjacent LSOAs (each polygon shrinks
# independently, leaving micro-gaps between neighbours), defeating Queen
# contiguity in libpysal. Keeping the full geometry costs ~17 MB but guarantees
# correct topology for spatial weights.

GEOJSON_OUT = CLEAN_DIR / "london_lsoa.geojson"

lsoa_full = gpd.read_file(LSOA_SHP)

# Keep identifiers (LSOA + MSOA + LAD) + geometry; drop population/density columns
# (we use IMD 2019 population denominator instead — more authoritative).
# MSOA columns are required for Section 1.7 MAUP / multi-scale analysis.
lsoa_slim = lsoa_full[[
    "LSOA11CD", "LSOA11NM",
    "MSOA11CD", "MSOA11NM",
    "LAD11CD",  "LAD11NM",
    "geometry",
]].copy()

# Reproject to WGS84 (GeoJSON convention)
lsoa_slim = lsoa_slim.to_crs(epsg=4326)

lsoa_slim.to_file(GEOJSON_OUT, driver="GeoJSON")

print(f"Wrote: {GEOJSON_OUT}")
print(f"LSOA polygons: {len(lsoa_slim):,}")
print(f"Unique MSOAs:  {lsoa_slim['MSOA11CD'].nunique():,}")
print(f"Unique LADs:   {lsoa_slim['LAD11CD'].nunique():,}")
print(f"File size: {GEOJSON_OUT.stat().st_size/1e6:.2f} MB")

Wrote: c:\Users\FanYuxiang\Desktop\casa0006-vru-london\data\clean\london_lsoa.geojson
LSOA polygons: 4,835
Unique MSOAs:  983
Unique LADs:   33
File size: 8.25 MB


## Step 9 — Export slim IMD 2019 (London only)

`analysis.ipynb` needs IMD 2019 for two purposes: **(a)** the `Total population` column as denominator for LSOA-level incidence rates (Phase 3 ESDA), and **(b)** the `IMD Score` as a deprivation control variable in Phase 4 Logistic regression. The original CSV (32,844 LSOAs, England-wide, 9 MB) is gitignored. Here we filter to London's 4,835 LSOAs, keep 7 relevant columns, and rename them to snake_case for clean downstream code.

In [10]:
# Export slim IMD 2019 (London only)
# Used by analysis.ipynb as deprivation control + population denominator.

IMD_OUT = CLEAN_DIR / "imd2019_london.csv"

# Load full England IMD 2019 (File 7: scores + ranks + deciles + population denominators)
imd_full = pd.read_csv(IMD_CSV)
print(f"IMD 2019 (full England): {len(imd_full):,} LSOAs × {len(imd_full.columns)} cols")

# Map gov.uk's verbose column names to clean snake_case for analysis.ipynb
col_map = {
    "LSOA code (2011)":                                                                "LSOA11CD",
    "Total population: mid 2015 (excluding prisoners)":                                "total_population",
    "Index of Multiple Deprivation (IMD) Score":                                       "imd_score",
    "Index of Multiple Deprivation (IMD) Decile (where 1 is most deprived 10% of LSOAs)": "imd_decile",
    "Income Score (rate)":                                                             "income_score",
    "Health Deprivation and Disability Score":                                         "health_score",
    "Crime Score":                                                                     "crime_score",
}

# Verify all expected columns exist (fail-fast on schema drift)
missing = [c for c in col_map if c not in imd_full.columns]
assert not missing, f"IMD missing columns: {missing}\nActual columns: {list(imd_full.columns)}"

# Slim + rename
imd_slim = imd_full[list(col_map.keys())].rename(columns=col_map)

# Filter to London LSOAs only (uses `london_lsoa_codes` from Step 3)
imd_lon = imd_slim[imd_slim["LSOA11CD"].isin(london_lsoa_codes)].copy()

# Sanity: should match the 4,835 London LSOAs
assert len(imd_lon) == len(london_lsoa_codes), (
    f"IMD London LSOAs ({len(imd_lon)}) ≠ shapefile London LSOAs ({len(london_lsoa_codes)})"
)

# Write
imd_lon.to_csv(IMD_OUT, index=False)

print(f"Wrote: {IMD_OUT}")
print(f"London LSOAs: {len(imd_lon):,}")
print(f"File size: {IMD_OUT.stat().st_size/1024:.1f} KB")
print(f"\nColumns: {list(imd_lon.columns)}")
print(f"\nSample row:\n{imd_lon.iloc[0]}")

IMD 2019 (full England): 32,844 LSOAs × 57 cols
Wrote: c:\Users\FanYuxiang\Desktop\casa0006-vru-london\data\clean\imd2019_london.csv
London LSOAs: 4,835
File size: 205.1 KB

Columns: ['LSOA11CD', 'total_population', 'imd_score', 'imd_decile', 'income_score', 'health_score', 'crime_score']

Sample row:
LSOA11CD            E01000001
total_population         1296
imd_score               6.208
imd_decile                  9
income_score            0.007
health_score           -1.654
crime_score            -2.012
Name: 0, dtype: object
